In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Ensure timestamp numeric integer where present
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')

# Helper: epoch seconds -> Asia/Shanghai localized string
def epoch_to_shanghai_str(s):
    if pd.isna(s):
        return pd.NA
    return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# Aggregation helper to compute requested summary and rename columns exactly as asked
def aggregate_series(df, group_cols, value_col_name):
    if df.empty:
        cols = group_cols + ['count(points)', 'min_timestamp', 'max_timestamp', 'p50(value)', 'p95(value)', 'max(value)']
        return pd.DataFrame(columns=cols)
    grp = df.groupby(group_cols, as_index=False).agg(
        **{
            'count(points)': (value_col_name, 'count'),
            'min_timestamp': ('timestamp', 'min'),
            'max_timestamp': ('timestamp', 'max'),
            'p50(value)': (value_col_name, lambda x: float(np.percentile(x.dropna(), 50)) if x.dropna().size>0 else np.nan),
            'p95(value)': (value_col_name, lambda x: float(np.percentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan),
            'max(value)': (value_col_name, 'max')
        }
    )
    # Convert timestamps to Shanghai strings
    grp['min_timestamp'] = grp['min_timestamp'].apply(lambda x: epoch_to_shanghai_str(x) if pd.notna(x) else pd.NA)
    grp['max_timestamp'] = grp['max_timestamp'].apply(lambda x: epoch_to_shanghai_str(x) if pd.notna(x) else pd.NA)
    # Sort for compactness: by count desc then group cols asc
    sort_by = ['count(points)'] + group_cols
    grp = grp.sort_values(['count(points)'] + group_cols, ascending=[False] + [True]*len(group_cols)).reset_index(drop=True)
    return grp

# 1) metric.csv: group by cmdb_id and kpi_name
metric_summary = aggregate_series(metric, ['cmdb_id', 'kpi_name'], 'value').head(50)

# 2) trace.csv: group by cmdb_id and trace_name
trace_summary = aggregate_series(trace, ['cmdb_id', 'trace_name'], 'value').head(50)

# 3) log.csv: group by cmdb_id and log_name
log_summary = aggregate_series(log, ['cmdb_id', 'log_name'], 'value').head(50)

# 4) error_logs.csv summary
if not error_logs.empty:
    total_error_rows = int(len(error_logs))
    min_err = int(error_logs['timestamp'].min())
    max_err = int(error_logs['timestamp'].max())
    error_info = pd.DataFrame([{
        'total_rows': total_error_rows,
        'min_timestamp': epoch_to_shanghai_str(min_err),
        'max_timestamp': epoch_to_shanghai_str(max_err)
    }])
    error_top20 = error_logs.sort_values('timestamp', ascending=True)[['timestamp','cmdb_id','message']].head(20).copy()
    error_top20['timestamp'] = error_top20['timestamp'].apply(lambda x: epoch_to_shanghai_str(x) if pd.notna(x) else pd.NA)
else:
    error_info = pd.DataFrame([{'total_rows': 0, 'min_timestamp': pd.NA, 'max_timestamp': pd.NA}])
    error_top20 = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Overall min and max timestamp across all files
ts_list = []
for df in (metric, trace, log, error_logs):
    if not df.empty and 'timestamp' in df.columns:
        ts_list.append(df['timestamp'].dropna().astype('Int64'))
if ts_list:
    all_ts = pd.concat(ts_list, ignore_index=True)
    overall_min = int(all_ts.min())
    overall_max = int(all_ts.max())
    overall_range = pd.DataFrame([{
        'overall_min_timestamp': epoch_to_shanghai_str(overall_min),
        'overall_max_timestamp': epoch_to_shanghai_str(overall_max)
    }])
else:
    overall_range = pd.DataFrame([{'overall_min_timestamp': pd.NA, 'overall_max_timestamp': pd.NA}])

# Final compact outputs (limited rows where applicable)
metric_summary, trace_summary, log_summary, error_info, error_top20, overall_range

```
Out[1]:
```
```python
# Summarize previous aggregation results into a concise plain-English summary string.
# This cell reuses existing variables from prior execution: metric_summary, trace_summary, log_summary, error_info, error_top20, overall_range

def fmt_num(x):
    try:
        if pd.isna(x):
            return "<NA>"
        # format large ints without scientific notation, floats with 3 sig figs
        if float(x).is_integer():
            return str(int(float(x)))
        val = float(x)
        if abs(val) >= 1e6:
            return f"{val:,.0f}"
        if abs(val) >= 1:
            return f"{val:.3f}"
        return f"{val:.6f}"
    except Exception:
        return str(x)

lines = []
# Overall time range
if 'overall_range' in globals() and not overall_range.empty:
    om = overall_range.loc[0, 'overall_min_timestamp']
    ox = overall_range.loc[0, 'overall_max_timestamp']
    lines.append(f"Overall time range (Asia/Shanghai): {om}  to  {ox}")
else:
    lines.append("Overall time range: <no timestamps available>")

lines.append("")

# metric summary top 5
lines.append("metric.csv — top groups (by count, up to 5):")
if 'metric_summary' in globals() and not metric_summary.empty:
    for _, r in metric_summary.head(5).iterrows():
        lines.append(
            f"  - {r['cmdb_id']} | {r['kpi_name']} | points={fmt_num(r['count(points)'])} | "
            f"p50={fmt_num(r['p50(value)'])} | p95={fmt_num(r['p95(value)'])} | max={fmt_num(r['max(value)'])}"
        )
    lines.append(f"  (showing {min(5, len(metric_summary))} of {len(metric_summary)} grouped KPI rows)")
else:
    lines.append("  (no metric data)")

lines.append("")

# trace summary top 5
lines.append("trace.csv — top groups (by count, up to 5):")
if 'trace_summary' in globals() and not trace_summary.empty:
    for _, r in trace_summary.head(5).iterrows():
        lines.append(
            f"  - {r['cmdb_id']} | {r['trace_name']} | points={fmt_num(r['count(points)'])} | "
            f"p50={fmt_num(r['p50(value)'])} | p95={fmt_num(r['p95(value)'])} | max={fmt_num(r['max(value)'])}"
        )
    lines.append(f"  (showing {min(5, len(trace_summary))} of {len(trace_summary)} grouped trace rows)")
else:
    lines.append("  (no trace data)")

lines.append("")

# log summary top 5
lines.append("log.csv — top groups (by count, up to 5):")
if 'log_summary' in globals() and not log_summary.empty:
    for _, r in log_summary.head(5).iterrows():
        lines.append(
            f"  - {r['cmdb_id']} | {r['log_name']} | points={fmt_num(r['count(points)'])} | "
            f"p50={fmt_num(r['p50(value)'])} | p95={fmt_num(r['p95(value)'])} | max={fmt_num(r['max(value)'])}"
        )
    lines.append(f"  (showing {min(5, len(log_summary))} of {len(log_summary)} grouped log rows)")
else:
    lines.append("  (no log data)")

lines.append("")

# error_logs summary and top rows notice
if 'error_info' in globals() and not error_info.empty:
    total = int(error_info.loc[0, 'total_rows'])
    min_ts = error_info.loc[0, 'min_timestamp']
    max_ts = error_info.loc[0, 'max_timestamp']
    lines.append(f"error_logs.csv — total rows: {total}; time range: {min_ts}  to  {max_ts}")
    if total > 0:
        lines.append("Top error_logs rows (up to 20) — showing timestamp, cmdb_id, message for first rows:")
        # show up to 5 here to keep compact, mention more available
        topn = min(5, len(error_top20))
        for _, r in error_top20.head(topn).iterrows():
            lines.append(f"  - {r['timestamp']} | {r['cmdb_id']} | {r['message']}")
        if total > topn:
            lines.append(f"  ... (showing {topn} of up to 20 top rows)")
else:
    lines.append("error_logs.csv — no error log metadata available")

lines.append("")
lines.append("Notes:")
lines.append(" - Percentiles (p50/p95) were computed on full KPI/trace/log value series before any time filtering.")
lines.append(" - All timestamps are shown in Asia/Shanghai (UTC+8).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name  count(points)            min_timestamp            max_timestamp    p50(value)    p95(value)    max(value)
0      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST  2.040136e+09  2.040136e+09  2.040136e+09
1      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST  4.464512e+08  8.187404e+08  8.433882e+08
2      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST  5.791533e+07  5.791533e+07  5.791533e+07
3      IG01           JVM-Threads_7778_JVM_ThreadCount_Threads             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST  6.900000e+01  7.400000e+01  7.500000e+01
4      IG01                            OSLinux-CPU_CPU_CPULoad             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST  1.060000e+00  1.365000e+00  1.430000e+00
..      ...                                                ...            ...                      ...                      ...           ...           ...           ...
45  Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST  0.000000e+00  0.000000e+00  0.000000e+00
46  Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST  4.000000e-01  4.000000e-01  4.000000e-01
47  Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST  7.000000e+00  7.000000e+00  7.000000e+00
48  Mysql01                 OSLinux-OSLinux_ZABBIX_Host_Uptime             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST  6.474477e+06  6.475289e+06  6.475379e+06
49  Mysql02                            OSLinux-CPU_CPU_CPULoad             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST  1.510000e+00  2.070000e+00  2.190000e+00

[50 rows x 8 columns],    cmdb_id                         trace_name  count(points)            min_timestamp            max_timestamp   p50(value)   p95(value)   max(value)
0     IG01           trace.self.duration_mean             30  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST     0.616591     1.005696     1.579263
1     IG01            trace.self.duration_p95             30  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST     2.682450     4.176140     6.495000
2     IG01               trace.self.row_count             30  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST  1944.000000  2416.900000  2448.000000
3     IG01    trace.to_Tomcat01.duration_mean             30  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST     0.606604     1.006720     1.645598
4     IG01     trace.to_Tomcat01.duration_p95             30  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST     2.236225     4.190285     6.499750
..     ...                                ...            ...                      ...                      ...          ...          ...          ...
45    MG01  trace.from_dockerA2.duration_mean             30  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST     0.060498     0.309139     0.446736
46    MG01   trace.from_dockerA2.duration_p95             30  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST     0.190800     1.507415     2.552000
47    MG01      trace.from_dockerA2.row_count             30  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST    40.500000    74.750000    81.000000
48    MG01  trace.from_dockerB1.duration_mean             30  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST     0.078730     0.293606     0.401754
49    MG01   trace.from_dockerB1.duration_p95             30  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST     1.012000     1.304595     1.856100

[50 rows x 8 columns],      cmdb_id          log_name  count(points)            min_timestamp            max_timestamp  p50(value)  p95(value)  max(value)
0   Tomcat01   log.error_count             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST       0.000      0.0000        0.00
1   Tomcat01     log.row_count             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST     486.000    581.0000      634.00
2   Tomcat03   log.error_count             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST       0.000      0.0000        0.00
3   Tomcat03     log.row_count             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST     478.000    581.0000      657.00
4   Tomcat04   log.error_count             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST       0.000      0.0000        0.00
5   Tomcat04     log.row_count             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST     476.000    594.5000      635.00
6   apache01   log.error_count             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST       0.000      0.0000        0.00
7   apache01     log.row_count             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST     962.000   1181.0000     1213.00
8   apache02   log.error_count             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST       0.000      0.0000        0.00
9   apache02     log.row_count             31  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST     969.000   1161.5000     1207.00
10   Mysql01   log.error_count             30  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST       0.000      0.0000        0.00
11   Mysql01     log.row_count             30  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST       7.000     22.0000       32.00
12  Tomcat01  log.gc_pause_max             24  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST       0.065      0.0900        0.10
13  Tomcat03  log.gc_pause_max             24  2021-03-09 15:30:00 CST  2021-03-09 15:58:00 CST       0.070      0.0900        0.12
14  Tomcat04  log.gc_pause_max             24  2021-03-09 15:30:00 CST  2021-03-09 15:59:00 CST       0.060      0.0985        0.12
15  Tomcat02   log.error_count             21  2021-03-09 15:30:00 CST  2021-03-09 15:58:00 CST       0.000      0.0000        0.00
16  Tomcat02  log.gc_pause_max             21  2021-03-09 15:30:00 CST  2021-03-09 15:58:00 CST       0.070      0.1100        0.11
17  Tomcat02     log.row_count             21  2021-03-09 15:30:00 CST  2021-03-09 15:58:00 CST       1.000      1.0000        1.00,    total_rows min_timestamp max_timestamp
0           0          <NA>          <NA>, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [],      overall_min_timestamp    overall_max_timestamp
0  2021-03-09 15:30:00 CST  2021-03-09 16:00:00 CST)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing 'metric' DataFrame loaded previously.
# Ensure timestamp and value types are numeric
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce').astype('Int64')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Timezone for Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Incident window boundaries (inclusive) in Asia/Shanghai -> convert to epoch seconds (UTC)
start_ts = int(pd.Timestamp("2021-03-09 15:30:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2021-03-09 16:00:00", tz=tz).tz_convert('UTC').timestamp())

# 1) Compute global P95 and total points per group using full series (no filtering)
group_stats = metric.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    total_points_full_series = ('value', 'count'),
    p95_global = ('value', lambda x: float(np.percentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan)
)

# 2) Filter metric to incident window
metric_window = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# Join global p95 onto window rows for comparison
metric_window = metric_window.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

# Determine if each row in window is above the group's global p95
metric_window['above_p95'] = metric_window['value'] > metric_window['p95_global']

# Aggregations per group within the window
agg_window = metric_window.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    count_points_in_window = ('value', 'count'),
    count_points_above_p95 = ('above_p95', 'sum'),
    max_value_in_window = ('value', 'max'),
    mean_value_in_window = ('value', 'mean')
)

# Earliest timestamp where above_p95 within window (compute separately)
mask_above = metric_window[metric_window['above_p95']].copy()
earliest_above = mask_above.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    earliest_timestamp_above_p95 = ('timestamp', 'min')
)

# Merge earliest timestamp into window agg
agg_window = agg_window.merge(earliest_above, on=['cmdb_id','kpi_name'], how='left')

# Merge with global stats to include total_points_full_series and p95_global
agg_full = agg_window.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

# Convert earliest timestamp to Asia/Shanghai readable string, keep NaN as <NA>
def epoch_to_shanghai_str_safe(s):
    if pd.isna(s):
        return pd.NA
    return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

agg_full['earliest_timestamp_above_p95'] = agg_full['earliest_timestamp_above_p95'].apply(epoch_to_shanghai_str_safe)

# 3) Filter groups where count_points_above_p95 > 0
anomalies = agg_full[agg_full['count_points_above_p95'] > 0].copy()

# Keep and order requested columns
anomalies = anomalies[
    ['cmdb_id','kpi_name','total_points_full_series','p95_global',
     'count_points_in_window','count_points_above_p95','earliest_timestamp_above_p95',
     'max_value_in_window','mean_value_in_window']
]

# Sort by count_points_above_p95 desc then max_value_in_window desc
anomalies = anomalies.sort_values(
    by=['count_points_above_p95','max_value_in_window'],
    ascending=[False, False]
).reset_index(drop=True)

anomalies_top20 = anomalies.head(20)

# 4) Counts: total distinct groups scanned and how many had anomalies
groups_scanned = int(group_stats.shape[0])
groups_with_anomalies_count = int(anomalies.shape[0])

# Keep intermediate full tables in kernel variables for later steps:
group_stats_full = group_stats        # global p95 and total_points_full_series per group
metric_window_full = metric_window    # windowed rows with p95 and above flag
agg_window_full = agg_full            # aggregated window results merged with global stats

# Final compact outputs
anomalies_top20, groups_scanned, groups_with_anomalies_count

```
Out[2]:
```
```python
# Build a concise plain-English summary from previous results stored in kernel variables.
# Reuse existing variables: anomalies_top20, groups_scanned, groups_with_anomalies_count

def fmt_num(v):
    try:
        if pd.isna(v):
            return "<NA>"
        v = float(v)
        # compact formatting: use integer if whole, otherwise 3 significant digits
        if abs(v - round(v)) < 1e-9:
            return str(int(round(v)))
        return f"{v:.3g}"
    except Exception:
        return str(v)

lines = []
if 'groups_scanned' in globals() and 'groups_with_anomalies_count' in globals():
    lines.append(f"Scanned groups (component-KPI): {groups_scanned}")
    lines.append(f"Groups with anomalies in window (value > global P95): {groups_with_anomalies_count}")
else:
    lines.append("Group counts not available.")

lines.append("")
lines.append("Incident window: 2021-03-09 15:30:00 CST  to  2021-03-09 16:00:00 CST (Asia/Shanghai)")
lines.append("Anomalies shown below are groups that had at least one point above their global P95.")

# Show top 5 example anomaly groups (compact)
if 'anomalies_top20' in globals() and not anomalies_top20.empty:
    lines.append("")
    lines.append("Top anomaly examples (up to 5):")
    for _, r in anomalies_top20.head(5).iterrows():
        cmdb = r['cmdb_id']
        kpi = (r['kpi_name'][:120] + '...') if len(str(r['kpi_name']))>120 else r['kpi_name']
        total = fmt_num(r['total_points_full_series'])
        p95 = fmt_num(r['p95_global'])
        cnt_win = fmt_num(r['count_points_in_window'])
        cnt_above = fmt_num(r['count_points_above_p95'])
        first_ts = r['earliest_timestamp_above_p95'] if pd.notna(r['earliest_timestamp_above_p95']) else "<NA>"
        maxv = fmt_num(r['max_value_in_window'])
        meanv = fmt_num(r['mean_value_in_window'])
        lines.append(f" - {cmdb} | {kpi}")
        lines.append(f"     total_pts_full={total}, p95_global={p95}, pts_in_window={cnt_win}, pts_above_p95={cnt_above}")
        lines.append(f"     earliest_above={first_ts}, max_in_window={maxv}, mean_in_window={meanv}")
else:
    lines.append("")
    lines.append("No anomaly groups found in the window.")

lines.append("")
lines.append("Interpretation:")
lines.append(" - A large number of groups exceeded their global P95 inside the incident window,")
lines.append("   indicating widespread elevated metric values (examples above).")
lines.append(" - Investigate the listed component-KPIs (Redis and JVM memory and Tomcat request metrics appear among top hits).")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  total_points_full_series    p95_global  count_points_in_window  count_points_above_p95 earliest_timestamp_above_p95  max_value_in_window  mean_value_in_window
0    Redis01  redis-Redis_6379_Redis  (total_commands_proces...                        31  4.990538e+09                      31                       2      2021-03-09 15:59:00 CST         4.990930e+09          4.987026e+09
1    Redis02  redis-Redis_6379_Redis  (total_commands_proces...                        31  3.972218e+09                      31                       2      2021-03-09 15:59:00 CST         3.972449e+09          3.970055e+09
2    Redis01            redis-Redis_6379_Redis  (keyspace_hits)                        31  3.791347e+09                      31                       2      2021-03-09 15:59:00 CST         3.791663e+09          3.788511e+09
3    Redis02            redis-Redis_6379_Redis  (keyspace_hits)                        31  2.870743e+09                      31                       2      2021-03-09 15:59:00 CST         2.870919e+09          2.869091e+09
4       MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed                        31  1.170341e+09                      31                       2      2021-03-09 15:39:00 CST         1.195176e+09          7.883952e+08
5    Redis01  redis-Redis_6379_Redis  (rdb_changes_since_las...                        31  1.103407e+09                      31                       2      2021-03-09 15:59:00 CST         1.103481e+09          1.102748e+09
6       MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed                        31  1.022277e+09                      31                       2      2021-03-09 15:30:00 CST         1.062335e+09          6.589587e+08
7   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...                        31  9.236032e+08                      31                       2      2021-03-09 15:59:00 CST         9.239711e+08          9.199382e+08
8    Redis02  redis-Redis_6379_Redis  (rdb_changes_since_las...                        31  8.929014e+08                      31                       2      2021-03-09 15:59:00 CST         8.929548e+08          8.924014e+08
9       IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed                        31  8.419098e+08                      31                       2      2021-03-09 15:35:00 CST         8.760316e+08          4.770784e+08
10      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed                        31  8.187404e+08                      31                       2      2021-03-09 15:42:00 CST         8.433882e+08          4.539119e+08
11  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...                        31  1.588547e+08                      31                       2      2021-03-09 15:59:00 CST         1.591993e+08          1.552630e+08
12  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...                        31  1.587001e+08                      31                       2      2021-03-09 15:59:00 CST         1.590840e+08          1.552568e+08
13  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...                        31  1.459153e+08                      31                       2      2021-03-09 15:59:00 CST         1.462192e+08          1.424174e+08
14   Redis02  redis-Redis_6379_Redis  (total_connections_rec...                        31  7.282905e+07                      31                       2      2021-03-09 15:59:00 CST         7.282956e+07          7.282549e+07
15   Redis01  redis-Redis_6379_Redis  (total_connections_rec...                        31  3.202547e+07                      31                       2      2021-03-09 15:59:00 CST         3.202612e+07          3.202005e+07
16   Redis01              redis-Redis_6379_Redis  (used_memory)                        31  1.836301e+07                      31                       2      2021-03-09 15:37:00 CST         2.204907e+07          1.808977e+07
17   Redis02              redis-Redis_6379_Redis  (used_memory)                        31  1.343646e+07                      31                       2      2021-03-09 15:36:00 CST         1.349783e+07          1.333070e+07
18   Redis01          redis-Redis_6379_Redis  (used_memory_rss)                        31  1.266483e+07                      31                       2      2021-03-09 15:57:00 CST         1.282867e+07          1.243625e+07
19   Redis02          redis-Redis_6379_Redis  (used_memory_rss)                        31  9.345024e+06                      31                       2      2021-03-09 15:49:00 CST         9.355264e+06          9.289332e+06, 1709, 638)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables from kernel:
# 'anomalies' : full anomalies table (groups with count_points_above_p95 > 0)
# 'metric_window_full' : windowed rows with 'above_p95' flag and numeric 'timestamp'
# 'tz' may exist; ensure timezone object
tz = pytz.timezone('Asia/Shanghai')

# Safety checks
if 'anomalies' not in globals() or anomalies is None or anomalies.empty:
    raise ValueError("Expected 'anomalies' DataFrame in kernel (non-empty).")

if 'metric_window_full' not in globals() or metric_window_full is None:
    raise ValueError("Expected 'metric_window_full' DataFrame in kernel.")

# Ensure numeric types for relevant columns
anomalies['p95_global'] = pd.to_numeric(anomalies['p95_global'], errors='coerce')
anomalies['max_value_in_window'] = pd.to_numeric(anomalies['max_value_in_window'], errors='coerce')
anomalies['count_points_above_p95'] = pd.to_numeric(anomalies['count_points_above_p95'], errors='coerce')
anomalies['count_points_in_window'] = pd.to_numeric(anomalies['count_points_in_window'], errors='coerce')

# 1) Compute exceed ratio per KPI row
anomalies = anomalies.copy()
anomalies['exceed_ratio'] = anomalies['max_value_in_window'] / anomalies['p95_global']

# 1a) Earliest anomaly timestamp per component using metric_window_full (numeric epoch)
# metric_window_full must contain 'timestamp', 'cmdb_id', and 'above_p95'
metric_window_full['timestamp'] = pd.to_numeric(metric_window_full['timestamp'], errors='coerce').astype('Int64')
metric_window_full['above_p95'] = metric_window_full['above_p95'].astype(bool)

earliest_per_comp = (
    metric_window_full[metric_window_full['above_p95']]
    .groupby('cmdb_id', as_index=False)
    .agg(earliest_anom_epoch=('timestamp', 'min'))
)

# Convert epoch -> Asia/Shanghai string
def epoch_to_shanghai_str_safe(s):
    if pd.isna(s):
        return pd.NA
    return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

earliest_per_comp['earliest_anomaly_timestamp'] = earliest_per_comp['earliest_anom_epoch'].apply(epoch_to_shanghai_str_safe)
earliest_per_comp = earliest_per_comp.drop(columns=['earliest_anom_epoch'])

# 1b) For each component compute required aggregates from 'anomalies' table
grouped = anomalies.groupby('cmdb_id', as_index=False).agg(
    num_anomalous_kpis = ('kpi_name', 'nunique'),
    total_anomaly_points_in_window = ('count_points_above_p95', 'sum')
)

# Compute max_exceed_ratio and sample_top_kpi per component
idx_max_ratio = anomalies.groupby('cmdb_id')['exceed_ratio'].idxmax()
top_ratio_rows = anomalies.loc[idx_max_ratio].reset_index(drop=True)[['cmdb_id','kpi_name','exceed_ratio']].rename(
    columns={'kpi_name':'sample_top_kpi','exceed_ratio':'max_exceed_ratio'}
)

# Merge aggregates
component_agg_full = grouped.merge(top_ratio_rows, on='cmdb_id', how='left').merge(
    earliest_per_comp, left_on='cmdb_id', right_on='cmdb_id', how='left'
)

# Rename column to component_name
component_agg_full = component_agg_full.rename(columns={'cmdb_id':'component_name'})

# Reorder columns
component_agg_full = component_agg_full[
    ['component_name','num_anomalous_kpis','total_anomaly_points_in_window',
     'earliest_anomaly_timestamp','max_exceed_ratio','sample_top_kpi']
]

# Format max_exceed_ratio as float (keep raw in DF)
component_agg_full['max_exceed_ratio'] = pd.to_numeric(component_agg_full['max_exceed_ratio'], errors='coerce')

# 2) Top 10 components sorted by num_anomalous_kpis desc, then total_anomaly_points_in_window desc
component_summary_top10 = component_agg_full.sort_values(
    by=['num_anomalous_kpis','total_anomaly_points_in_window'],
    ascending=[False, False]
).reset_index(drop=True).head(10)

# 3) For single top-ranked component, list its top 10 anomalous KPIs sorted by count_points_above_p95 desc
if not component_summary_top10.empty:
    top_component = component_summary_top10.loc[0, 'component_name']
    top_component_kpis_full = anomalies[anomalies['cmdb_id'] == top_component].copy()
    top_component_kpis = top_component_kpis_full.sort_values(
        by=['count_points_above_p95','count_points_in_window'], ascending=[False, False]
    ).reset_index(drop=True)[
        ['kpi_name','count_points_in_window','count_points_above_p95','earliest_timestamp_above_p95','p95_global','max_value_in_window']
    ].head(10)
else:
    top_component = None
    top_component_kpis = pd.DataFrame(columns=['kpi_name','count_points_in_window','count_points_above_p95','earliest_timestamp_above_p95','p95_global','max_value_in_window'])

# Store full intermediates for next steps
component_agg_full_df = component_agg_full     # full aggregation by component
top_component_kpis_full_df = top_component_kpis_full if 'top_component_kpis_full' in locals() else pd.DataFrame()

# Final compact outputs
component_summary_top10, top_component_kpis

```
Out[3]:
```
```python
# Summarize the aggregation results into a concise plain-English summary string.
# Reuse existing kernel variables: component_summary_top10, top_component_kpis

def fmt(v):
    import math
    if v is None or (isinstance(v, float) and (math.isnan(v))):
        return "<NA>"
    if isinstance(v, (int,)) or (isinstance(v, float) and abs(v - round(v)) < 1e-9):
        return str(int(round(v)))
    try:
        if abs(float(v)) >= 1e6:
            return f"{float(v):,.0f}"
        return f"{float(v):.6g}"
    except Exception:
        return str(v)

lines = []
lines.append("Component-level anomaly summary (compact):")
lines.append("Top components by number of anomalous KPIs:")

# component_summary_top10 exists from prior execution
if 'component_summary_top10' in globals() and not component_summary_top10.empty:
    for i, row in component_summary_top10.reset_index(drop=True).iterrows():
        lines.append(
            f"{i+1}. {row['component_name']}: anomalous_kpis={fmt(row['num_anomalous_kpis'])}, "
            f"anomaly_points={fmt(row['total_anomaly_points_in_window'])}, "
            f"earliest_anomaly={row['earliest_anomaly_timestamp']}, "
            f"max_exceed_ratio={fmt(row['max_exceed_ratio'])}, sample_top_kpi={row['sample_top_kpi']}"
        )
else:
    lines.append("  (no component summary available)")

lines.append("")
# Details for top-ranked component
if 'component_summary_top10' in globals() and not component_summary_top10.empty:
    top_comp = component_summary_top10.loc[0, 'component_name']
    lines.append(f"Top-ranked component: {top_comp}")
    lines.append("Top anomalous KPIs for this component (top 10 by points above P95):")
    if 'top_component_kpis' in globals() and not top_component_kpis.empty:
        for _, r in top_component_kpis.iterrows():
            lines.append(
                f" - KPI: {r['kpi_name']}\n"
                f"     pts_in_window={fmt(r['count_points_in_window'])}, pts_above_p95={fmt(r['count_points_above_p95'])}, "
                f"earliest_above={r['earliest_timestamp_above_p95']}, p95_global={fmt(r['p95_global'])}, "
                f"max_in_window={fmt(r['max_value_in_window'])}"
            )
    else:
        lines.append("  (no KPI details available for top component)")
else:
    lines.append("No top component available.")

lines.append("")
lines.append("Notes:")
lines.append(" - 'max_exceed_ratio = inf' indicates the group's global P95 was zero (division by zero).")
lines.append(" - All timestamps are Asia/Shanghai (CST, UTC+8).")
summary = "\n".join(lines)

# Display the concise summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  component_name  num_anomalous_kpis  total_anomaly_points_in_window earliest_anomaly_timestamp  max_exceed_ratio                                   sample_top_kpi
0        Mysql01                  88                              94    2021-03-09 15:30:00 CST         20.000000       Mysql-MySQL_3306_max trx lock memory bytes
1        Mysql02                  86                              93    2021-03-09 15:30:00 CST               inf             Mysql-MySQL_3306_Max trx rows locked
2       Tomcat01                  46                              49    2021-03-09 15:30:00 CST         10.000000   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT
3       Tomcat04                  44                              48    2021-03-09 15:30:00 CST          1.818182  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead
4           IG02                  43                              52    2021-03-09 15:30:00 CST               inf   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT
5           IG01                  42                              50    2021-03-09 15:30:00 CST         16.800000     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
6       Tomcat03                  41                              43    2021-03-09 15:31:00 CST          1.538462  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps
7        Redis01                  35                              53    2021-03-09 15:30:00 CST          1.466682                           OSLinux-CPU_CPU_CPUWio
8        Redis02                  35                              53    2021-03-09 15:30:00 CST          2.601757                           OSLinux-CPU_CPU_CPUWio
9       Tomcat02                  33                              36    2021-03-09 15:30:00 CST          2.000000   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps,                                  kpi_name  count_points_in_window  count_points_above_p95 earliest_timestamp_above_p95    p95_global  max_value_in_window
0      OSLinux-OSLinux_ZABBIX_Host_Uptime                      31                       2      2021-03-09 15:59:00 CST  6.475289e+06         6.475379e+06
1             OSLinux-CPU_CPU_CPUidleutil                      31                       2      2021-03-09 15:37:00 CST  9.843650e+01         9.861570e+01
2                  OSLinux-CPU_CPU_CPUWio                      31                       2      2021-03-09 15:49:00 CST  2.982900e+00         8.162600e+00
3              OSLinux-CPU_CPU_CPUSysTime                      31                       2      2021-03-09 15:35:00 CST  4.822665e-01         6.067000e-01
4             OSLinux-CPU_CPU_CPUUserTime                      31                       2      2021-03-09 15:35:00 CST  4.656000e-01         5.527000e-01
5                 OSLinux-CPU_CPU_CPULoad                      31                       2      2021-03-09 15:34:00 CST  3.500000e-01         5.200000e-01
6    Mysql-MySQL_3306_Innodb data written                      20                       1      2021-03-09 15:57:00 CST  3.085483e+05         3.156821e+05
7             Mysql-MySQL_3306_Bytes Sent                      20                       1      2021-03-09 15:38:00 CST  3.735154e+04         4.378828e+04
8  Mysql-MySQL_3306_Innodb os log written                      20                       1      2021-03-09 15:31:00 CST  4.504747e+03         4.650667e+03
9              Mysql-MySQL_3306_Rows Read                      20                       1      2021-03-09 15:40:00 CST  2.814364e+03         2.827200e+03)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing 'trace' DataFrame from kernel
# Ensure numeric types
trace['timestamp'] = pd.to_numeric(trace['timestamp'], errors='coerce').astype('Int64')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# Timezone and incident window (Asia/Shanghai UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp("2021-03-09 15:30:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2021-03-09 16:00:00", tz=tz).tz_convert('UTC').timestamp())

# Candidate components
candidates = ["Mysql01","Mysql02","Tomcat01","Tomcat04","IG02","IG01","Tomcat03","Redis01","Redis02","Tomcat02"]

# Helper: epoch -> Asia/Shanghai string
def epoch_to_shanghai_str_safe(s):
    if pd.isna(s):
        return pd.NA
    return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# 1) Compute global P95 and total points per (cmdb_id, trace_name) using full trace series
group_stats_trace_full = trace.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    total_points_full_series = ('value', 'count'),
    p95_global = ('value', lambda x: float(np.percentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan)
)

# 2) Filter trace rows to incident window
trace_window_full = trace[(trace['timestamp'] >= start_ts) & (trace['timestamp'] <= end_ts)].copy()

# Restrict to candidate components for further per-group window aggregations
trace_window_candidates = trace_window_full[trace_window_full['cmdb_id'].isin(candidates)].copy()

# Join global p95 onto window candidate rows
trace_window_candidates = trace_window_candidates.merge(
    group_stats_trace_full, on=['cmdb_id','trace_name'], how='left'
)

# Mark above global p95
trace_window_candidates['above_p95'] = trace_window_candidates['value'] > trace_window_candidates['p95_global']

# Aggregations per (cmdb_id, trace_name) within window for candidate components
agg_trace_window_full = trace_window_candidates.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    count_points_in_window = ('value', 'count'),
    count_points_above_p95 = ('above_p95', 'sum'),
    max_value_in_window = ('value', 'max')
)

# Earliest timestamp above p95 per group (epoch)
mask_above = trace_window_candidates[trace_window_candidates['above_p95']].copy()
earliest_above = mask_above.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    earliest_epoch_above_p95 = ('timestamp', 'min')
)

# Merge earliest into agg
agg_trace_window_full = agg_trace_window_full.merge(earliest_above, on=['cmdb_id','trace_name'], how='left')

# Merge p95_global for display
agg_trace_window_full = agg_trace_window_full.merge(
    group_stats_trace_full[['cmdb_id','trace_name','p95_global']],
    on=['cmdb_id','trace_name'],
    how='left'
)

# Convert earliest epoch to Asia/Shanghai string
agg_trace_window_full['earliest_timestamp_above_p95'] = agg_trace_window_full['earliest_epoch_above_p95'].apply(epoch_to_shanghai_str_safe)
agg_trace_window_full = agg_trace_window_full.drop(columns=['earliest_epoch_above_p95'])

# 3) Keep only groups where count_points_above_p95 > 0
anomalies_trace_full = agg_trace_window_full[agg_trace_window_full['count_points_above_p95'] > 0].copy()

# Sort by earliest_timestamp_above_p95 ascending (na last) then count_points_above_p95 descending
# For correct sorting, convert earliest timestamp back to epoch for sort if possible
def to_epoch_or_large_na(ts_str):
    if pd.isna(ts_str):
        return np.iinfo(np.int64).max
    try:
        return int(pd.to_datetime(ts_str).tz_localize(tz).tz_convert('UTC').timestamp())
    except Exception:
        # fallback
        return np.iinfo(np.int64).max

anomalies_trace_full['_earliest_epoch_for_sort'] = anomalies_trace_full['earliest_timestamp_above_p95'].apply(
    lambda x: int(pd.to_datetime(x).tz_convert('UTC').timestamp()) if pd.notna(x) and isinstance(x,str) and ' ' in x else np.iinfo(np.int64).max
) if False else None
# Simpler: compute sort using the original earliest_epoch where available by re-joining from mask_above
# Recompute earliest epoch map
earliest_map = mask_above.groupby(['cmdb_id','trace_name'], as_index=False).agg(earliest_epoch=('timestamp','min'))
anomalies_trace_full = anomalies_trace_full.merge(earliest_map, on=['cmdb_id','trace_name'], how='left')

anomalies_trace_full = anomalies_trace_full.sort_values(
    by=['earliest_epoch','count_points_above_p95'],
    ascending=[True, False]
).reset_index(drop=True)

# Prepare compact output columns
anomalies_trace_groups_top20 = anomalies_trace_full[
    ['cmdb_id','trace_name','p95_global','count_points_in_window','count_points_above_p95','earliest_timestamp_above_p95','max_value_in_window']
].head(20).copy()

# 4) Per-component summary for candidate list
# Compute per-component aggregates from anomalies_trace_full
comp_agg = anomalies_trace_full.groupby('cmdb_id', as_index=False).agg(
    num_anomalous_trace_groups = ('trace_name','nunique'),
    total_anomalous_trace_points = ('count_points_above_p95','sum'),
    earliest_epoch_trace_anomaly = ('earliest_epoch','min')
)

# Convert earliest epoch to string
comp_agg['earliest_trace_anomaly_timestamp'] = comp_agg['earliest_epoch_trace_anomaly'].apply(epoch_to_shanghai_str_safe)
comp_agg = comp_agg.drop(columns=['earliest_epoch_trace_anomaly'])

# Ensure all candidate components are present (include zeros for those without anomalies)
candidate_df = pd.DataFrame({'component_name': candidates})
comp_agg = candidate_df.merge(comp_agg.rename(columns={'cmdb_id':'component_name'}), on='component_name', how='left').fillna({
    'num_anomalous_trace_groups': 0,
    'total_anomalous_trace_points': 0,
    'earliest_trace_anomaly_timestamp': pd.NA
})

# Cast numeric fields to int
comp_agg['num_anomalous_trace_groups'] = comp_agg['num_anomalous_trace_groups'].astype(int)
comp_agg['total_anomalous_trace_points'] = comp_agg['total_anomalous_trace_points'].astype(int)

# Sort by num_anomalous_trace_groups desc
component_summary_candidate = comp_agg.sort_values(by=['num_anomalous_trace_groups','total_anomalous_trace_points'], ascending=[False, False]).reset_index(drop=True)

# Counts
# groups scanned = number of distinct (cmdb_id, trace_name) among candidate components in the full trace series
groups_scanned = int(trace[trace['cmdb_id'].isin(candidates)].drop_duplicates(['cmdb_id','trace_name']).shape[0])
anomalous_trace_groups_count = int(anomalies_trace_full.shape[0])

# Store intermediate tables for later steps
trace_group_stats_full = group_stats_trace_full
trace_window_candidates_full = trace_window_candidates
agg_trace_window_candidates_full = agg_trace_window_full
anomalies_trace_full_df = anomalies_trace_full
component_agg_full_df = comp_agg

# Final compact outputs
anomalies_trace_groups_top20, component_summary_candidate[['component_name','num_anomalous_trace_groups','total_anomalous_trace_points','earliest_trace_anomaly_timestamp']].head(50), groups_scanned, anomalous_trace_groups_count

```
Out[4]:
```
Summary of trace-level anomaly detection (candidate components) — plain English:

- Scope: scanned 108 distinct (component, trace_name) groups for the 10 candidate components during the incident window (2021-03-09 15:30 → 16:00 CST). Found 90 anomalous trace groups (value > group global P95).

- Primary impacted components: Tomcat04, IG02, IG01, Tomcat03, Tomcat02 each show 15 anomalous trace groups (total anomalous trace points = 30 for each), and Tomcat01 shows 15 anomalous groups with 29 anomalous points. These components exhibit the most widespread trace-level anomalies.

- Timing: earliest anomalous trace points for the top components are at 2021-03-09 15:30:00 CST (i.e., anomalies start at the beginning of the incident window).

- Example anomalous trace groups (top of the list):
  - IG01: trace.self.duration_mean (p95 ≈ 1.0057) — 30 points in window, 2 points above p95, earliest above at 15:30, max ≈ 1.5793.
  - IG01: trace.self.duration_p95 (p95 ≈ 4.1761) — 30 pts, 2 above, earliest 15:30, max ≈ 6.4950.
  - IG01: trace.to_Tomcat01.duration_mean and related to other Tomcat targets show similar patterns (2 points above p95, earliest 15:30).
  - IG02 and other Tomcat components show equivalent anomalies for many trace metrics.

- Not impacted (in this trace-level check): Mysql01, Mysql02, Redis01, Redis02 have zero anomalous trace groups in this window.

- Interpretation / next steps:
  - The anomalies are concentrated on IG and Tomcat components (latency/duration-related trace metrics), with anomalies appearing immediately at the start of the window and typically 1–2 points exceeding global P95 per trace group.
  - Focus investigation on IG and Tomcat services (latency traces: trace.self.duration*, trace.to_* duration/p95/row_count) to find common causes (upstream latency, increased request processing, or resource contention).
  - MySQL and Redis traces did not show trace-level anomalies in this window and are lower priority for trace-based root-cause investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                       trace_name  p95_global  count_points_in_window  count_points_above_p95 earliest_timestamp_above_p95  max_value_in_window
0     IG01         trace.self.duration_mean    1.005696                      30                       2      2021-03-09 15:30:00 CST             1.579263
1     IG01          trace.self.duration_p95    4.176140                      30                       2      2021-03-09 15:30:00 CST             6.495000
2     IG01  trace.to_Tomcat01.duration_mean    1.006720                      30                       2      2021-03-09 15:30:00 CST             1.645598
3     IG01   trace.to_Tomcat01.duration_p95    4.190285                      30                       2      2021-03-09 15:30:00 CST             6.499750
4     IG01  trace.to_Tomcat02.duration_mean    1.007995                      30                       2      2021-03-09 15:30:00 CST             1.648996
5     IG01   trace.to_Tomcat02.duration_p95    4.556450                      30                       2      2021-03-09 15:30:00 CST             6.364250
6     IG01  trace.to_Tomcat03.duration_mean    1.022871                      30                       2      2021-03-09 15:30:00 CST             1.247061
7     IG01   trace.to_Tomcat03.duration_p95    4.269135                      30                       2      2021-03-09 15:30:00 CST             5.149000
8     IG01  trace.to_Tomcat04.duration_mean    1.004549                      30                       2      2021-03-09 15:30:00 CST             1.712983
9     IG01   trace.to_Tomcat04.duration_p95    4.161510                      30                       2      2021-03-09 15:30:00 CST             7.949500
10    IG01      trace.to_Tomcat04.row_count  289.650000                      30                       2      2021-03-09 15:30:00 CST           320.000000
11    IG02         trace.self.duration_mean    1.014102                      30                       2      2021-03-09 15:30:00 CST             1.603357
12    IG02          trace.self.duration_p95    3.973942                      30                       2      2021-03-09 15:30:00 CST             6.888000
13    IG02  trace.to_Tomcat01.duration_mean    1.089707                      30                       2      2021-03-09 15:30:00 CST             1.605770
14    IG02   trace.to_Tomcat01.duration_p95    4.262857                      30                       2      2021-03-09 15:30:00 CST             6.291000
15    IG02  trace.to_Tomcat02.duration_mean    0.983093                      30                       2      2021-03-09 15:30:00 CST             1.645799
16    IG02   trace.to_Tomcat02.duration_p95    4.169672                      30                       2      2021-03-09 15:30:00 CST             6.107400
17    IG02  trace.to_Tomcat03.duration_mean    1.032544                      30                       2      2021-03-09 15:30:00 CST             1.447847
18    IG02   trace.to_Tomcat03.duration_p95    4.314987                      30                       2      2021-03-09 15:30:00 CST             6.521650
19    IG02  trace.to_Tomcat04.duration_mean    1.012429                      30                       2      2021-03-09 15:30:00 CST             1.681270,   component_name  num_anomalous_trace_groups  total_anomalous_trace_points earliest_trace_anomaly_timestamp
0       Tomcat04                          15                            30          2021-03-09 15:30:00 CST
1           IG02                          15                            30          2021-03-09 15:30:00 CST
2           IG01                          15                            30          2021-03-09 15:30:00 CST
3       Tomcat03                          15                            30          2021-03-09 15:30:00 CST
4       Tomcat02                          15                            30          2021-03-09 15:30:00 CST
5       Tomcat01                          15                            29          2021-03-09 15:30:00 CST
6        Mysql01                           0                             0                             <NA>
7        Mysql02                           0                             0                             <NA>
8        Redis01                           0                             0                             <NA>
9        Redis02                           0                             0                             <NA>, 108, 90)```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing 'metric' DataFrame from kernel
# Ensure numeric types
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce').astype('Int64')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Target Tomcat components
tomcats = ["Tomcat01","Tomcat02","Tomcat03","Tomcat04"]

# Timezone and incident window boundaries (Asia/Shanghai UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp("2021-03-09 15:30:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2021-03-09 16:00:00", tz=tz).tz_convert('UTC').timestamp())

# Helper: epoch -> Asia/Shanghai string
def epoch_to_shanghai_str_safe(s):
    if pd.isna(s):
        return pd.NA
    return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# 1) Compute global P95 and total points per (cmdb_id, kpi_name) for Tomcat components using full series
tomcat_full = metric[metric['cmdb_id'].isin(tomcats)].copy()

tomcat_group_stats_full = tomcat_full.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    total_points_full_series = ('value', 'count'),
    p95_global = ('value', lambda x: float(np.percentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan)
)

# 2) Filter metric to incident window and Tomcat components, join global stats
metric_tomcat_window = metric[
    (metric['cmdb_id'].isin(tomcats)) &
    (metric['timestamp'] >= start_ts) &
    (metric['timestamp'] <= end_ts)
].copy()

metric_tomcat_window = metric_tomcat_window.merge(
    tomcat_group_stats_full, on=['cmdb_id','kpi_name'], how='left'
)

# Mark rows above global p95 (handle NaN p95 as False)
metric_tomcat_window['above_p95'] = metric_tomcat_window.apply(
    lambda r: False if pd.isna(r['p95_global']) else (r['value'] > r['p95_global']),
    axis=1
)

# Aggregations per group within the window
tomcat_agg_window = metric_tomcat_window.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    count_points_in_window = ('value','count'),
    count_points_above_p95 = ('above_p95','sum'),
    max_value_in_window = ('value','max')
)

# Earliest timestamp above p95 per group
mask_above = metric_tomcat_window[metric_tomcat_window['above_p95']].copy()
earliest_above = mask_above.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    earliest_epoch_above_p95 = ('timestamp','min')
)

tomcat_agg_window = tomcat_agg_window.merge(earliest_above, on=['cmdb_id','kpi_name'], how='left')

# Merge p95_global for ratio calculation
tomcat_agg_window = tomcat_agg_window.merge(
    tomcat_group_stats_full[['cmdb_id','kpi_name','p95_global']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Compute exceed_ratio, handling p95_global == 0 or NaN
def compute_ratio(row):
    p = row['p95_global']
    m = row['max_value_in_window']
    if pd.isna(p):
        return np.nan
    try:
        if p == 0:
            if pd.isna(m):
                return np.nan
            return np.inf
        return m / p
    except Exception:
        return np.nan

tomcat_agg_window['exceed_ratio'] = tomcat_agg_window.apply(compute_ratio, axis=1)

# Convert earliest epoch to Asia/Shanghai string
tomcat_agg_window['earliest_timestamp_above_p95'] = tomcat_agg_window['earliest_epoch_above_p95'].apply(epoch_to_shanghai_str_safe)
tomcat_agg_window = tomcat_agg_window.drop(columns=['earliest_epoch_above_p95'])

# 3) Keep only groups where count_points_above_p95 > 0
tomcat_anomalies = tomcat_agg_window[tomcat_agg_window['count_points_above_p95'] > 0].copy()

# Sort by exceed_ratio desc then count_points_above_p95 desc
tomcat_anomalies = tomcat_anomalies.sort_values(
    by=['exceed_ratio','count_points_above_p95'],
    ascending=[False, False]
).reset_index(drop=True)

# Prepare compact output columns and limit to top 20
tomcat_anomalies_top20 = tomcat_anomalies[
    ['cmdb_id','kpi_name','p95_global','count_points_in_window','count_points_above_p95',
     'earliest_timestamp_above_p95','max_value_in_window','exceed_ratio']
].head(20).copy()

# 4) Counts: number of Tomcat groups scanned and number with anomalies
tomcat_groups_scanned = int(tomcat_group_stats_full.shape[0])
tomcat_groups_with_anomalies = int(tomcat_anomalies.shape[0])

# Store intermediates for later steps (kept in kernel variables)
tomcat_group_stats_full_df = tomcat_group_stats_full
metric_tomcat_window_df = metric_tomcat_window
tomcat_agg_window_df = tomcat_agg_window
tomcat_anomalies_full_df = tomcat_anomalies

# Final compact outputs
tomcat_anomalies_top20, tomcat_groups_scanned, tomcat_groups_with_anomalies

```
Out[5]:
```
Summary of Tomcat metric anomalies (incident window 2021-03-09 15:30 → 16:00 CST):

- Scope: scanned 437 distinct Tomcat (cmdb_id, kpi_name) groups. 164 groups showed at least one value above their global P95 in the incident window.

- Highest exceedances (examples, sorted by exceed_ratio):
  - Tomcat01 — OSLinux NETWORK TCP-CLOSE-WAIT: exceed_ratio = 10.0, 1 point above p95, earliest at 2021-03-09 15:41:00.
  - Tomcat02 — LOCALDISK sda DSKTps: exceed_ratio = 2.0, 1 point above p95, earliest at 2021-03-09 15:57:00.
  - Tomcat04 — LOCALDISK sdb DSKRTps and sdb DSKRead: exceed_ratio ≈ 1.82, each 1 point above p95, earliest at 2021-03-09 15:40:00.
  - Multiple Tomcat1 disk metrics (DSKBps/DSKTps/DSKRead/DSKWrite) show exceed_ratio ~1.5–1.8 with single-point spikes between ~15:39–15:50.
  - Tomcat02 — thread count metric had 2 points above p95 (exceed_ratio ≈ 1.6) starting at 15:30, indicating brief thread-count elevation.
  - Several single-CPU utilization metrics on Tomcat01 show spikes (exceed_ratio ~1.5–1.75) around 15:47–15:50.

- Pattern: Most anomalous groups are single or very few-point spikes (count_points_above_p95 = 1 for many KPIs; some KPIs have 2). The anomalies concentrate on:
  - Local disk I/O metrics (sda/sdb read/write/IOps/throughput) across Tomcat nodes.
  - Some CPU single-core utilization spikes and network TCP CLOSE-WAIT on Tomcat01.
  - A small number of Tomcat request/thread metrics (Tomcat02 threads) also exceeded P95.

- Timing: Many disk and CPU spikes occur ~15:39–15:50; some thread-related anomalies start at the beginning of the window (15:30).

- Actionable next steps:
  - Investigate disk I/O on Tomcat hosts (sda/sdb): check processes causing bursts, filesystem activity, and any background jobs or backups at ~15:40–15:50.
  - Inspect Tomcat01 network state (many TCP CLOSE-WAITs at 15:41) — could indicate slow peer connections or socket cleanup issues.
  - Correlate these metric spikes with trace-level anomalies (latency traces) and application logs around the same timestamps to narrow root cause.
  - Prioritize Tomcat01 and Tomcat04 for disk/network and Tomcat02 for thread-usage investigation.

If you want, I can produce a concise CSV-like table of the top anomalies shown above or expand details for specific Tomcat nodes or KPIs.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name    p95_global  count_points_in_window  count_points_above_p95 earliest_timestamp_above_p95  max_value_in_window  exceed_ratio
0   Tomcat01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT      0.100000                      19                       1      2021-03-09 15:41:00 CST               1.0000     10.000000
1   Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps      0.500000                      11                       1      2021-03-09 15:57:00 CST               1.0000      2.000000
2   Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps      0.009185                      10                       1      2021-03-09 15:40:00 CST               0.0167      1.818182
3   Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead      0.073315                      10                       1      2021-03-09 15:40:00 CST               0.1333      1.818182
4   Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps      1.100000                      10                       1      2021-03-09 15:44:00 CST               2.0000      1.818182
5   Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps      0.550000                      10                       1      2021-03-09 15:39:00 CST               1.0000      1.818182
6   Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...  61811.500000                      10                       1      2021-03-09 15:49:00 CST          112144.0000      1.814290
7   Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps     52.150000                      10                       1      2021-03-09 15:49:00 CST              94.0000      1.802493
8   Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps     16.400000                      10                       1      2021-03-09 15:48:00 CST              29.0000      1.768293
9   Tomcat01                    OSLinux-CPU_CPU-3_SingleCpuUtil     24.519450                      11                       1      2021-03-09 15:50:00 CST              43.0879      1.757295
10  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     10.000000                      31                       2      2021-03-09 15:30:00 CST              16.0000      1.600000
11  Tomcat02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...      0.130000                       8                       1      2021-03-09 15:56:00 CST               0.2000      1.538462
12  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps      0.021645                       8                       1      2021-03-09 15:37:00 CST               0.0333      1.538462
13  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps      3.900000                       8                       1      2021-03-09 15:41:00 CST               6.0000      1.538462
14  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead      0.173355                       8                       1      2021-03-09 15:37:00 CST               0.2667      1.538462
15  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead      2.386655                       8                       1      2021-03-09 15:48:00 CST               3.6000      1.508387
16  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps      0.298345                       8                       1      2021-03-09 15:48:00 CST               0.4500      1.508321
17  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps     62.316700                       8                       1      2021-03-09 15:48:00 CST              93.8167      1.505482
18  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  74533.380010                       8                       1      2021-03-09 15:48:00 CST          112141.0667      1.504575
19  Tomcat01                    OSLinux-CPU_CPU-0_SingleCpuUtil     37.219080                       9                       1      2021-03-09 15:47:00 CST              54.7512      1.471052, 437, 164)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing 'metric' DataFrame
# Ensure types
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce').astype('Int64')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Target components and timezone/window
components = ["IG01","IG02","Tomcat01","Tomcat02","Tomcat03","Tomcat04"]
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp("2021-03-09 15:30:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2021-03-09 16:00:00", tz=tz).tz_convert('UTC').timestamp())

# Helper to format epoch -> Asia/Shanghai string
def epoch_to_shanghai_str_safe(s):
    if pd.isna(s):
        return pd.NA
    return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# 1) Compute global P95 and total points per (cmdb_id, kpi_name) for these components (full series)
group_stats = (
    metric[metric['cmdb_id'].isin(components)]
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        total_points_full_series = ('value','count'),
        p95_global = ('value', lambda x: float(np.percentile(x.dropna(),95)) if x.dropna().size>0 else np.nan)
    )
)

# 2) Extract rows in incident window for these components and merge p95
metric_window = metric[
    (metric['cmdb_id'].isin(components)) &
    (metric['timestamp'] >= start_ts) &
    (metric['timestamp'] <= end_ts)
].copy()

metric_window = metric_window.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

# Mark anomaly points: value > p95_global (treat NaN p95 as False)
metric_window['anomaly'] = (~metric_window['p95_global'].isna()) & (metric_window['value'] > metric_window['p95_global'])

# 2b) For each group identify consecutive anomaly segments
def analyze_group(df):
    # df is windowed rows for a single (cmdb_id,kpi_name), unsorted -> sort by timestamp
    df = df.sort_values('timestamp').reset_index(drop=True)
    times = df['timestamp'].to_numpy(dtype='int64')
    above = df['anomaly'].to_numpy(dtype=bool)
    total_points_in_window = int(len(df))
    total_anomaly_points = int(above.sum())
    max_value_in_window = float(df['value'].max()) if not df['value'].dropna().empty else np.nan

    if total_anomaly_points == 0:
        return {
            'total_points_in_window': total_points_in_window,
            'total_anomaly_points': total_anomaly_points,
            'num_anomaly_segments': 0,
            'max_segment_length': 0,
            'earliest_segment_start_epoch': pd.NA,
            'max_value_in_window': max_value_in_window
        }
    # compute segment starts: an anomaly is a start if above and (prev not above or timestamp not consecutive)
    prev_above = np.concatenate(([False], above[:-1]))
    prev_time = np.concatenate(([-9999999999], times[:-1]))
    starts = above & (~prev_above | (prev_time + 60 != times))
    # segment ids: cumulative sum of starts, but zero where not above
    seg_id = np.cumsum(starts).astype(int) * above.astype(int)
    # consider only seg_id >0
    seg_ids = seg_id[seg_id>0]
    if seg_ids.size == 0:
        num_segments = 0
        max_seg_len = 0
        earliest_start_epoch = pd.NA
    else:
        # lengths per segment id
        unique_ids, counts = np.unique(seg_id[seg_id>0], return_counts=True)
        num_segments = int(unique_ids.size)
        max_seg_len = int(counts.max())
        # compute start epoch for each segment id: first timestamp where seg_id == id
        # find indices where seg_id >0 and where a new segment starts (starts True)
        start_indices = np.where(starts)[0]
        start_epochs = times[start_indices]
        # earliest start is min of start_epochs
        earliest_start_epoch = int(start_epochs.min())
    return {
        'total_points_in_window': total_points_in_window,
        'total_anomaly_points': total_anomaly_points,
        'num_anomaly_segments': num_segments,
        'max_segment_length': max_seg_len,
        'earliest_segment_start_epoch': earliest_start_epoch,
        'max_value_in_window': max_value_in_window
    }

# Apply analyze_group per group
grouped = metric_window.groupby(['cmdb_id','kpi_name'], sort=False)
results = []
for (cmdb,kpi), g in grouped:
    stats = analyze_group(g)
    stats.update({'cmdb_id':cmdb, 'kpi_name':kpi})
    # attach p95_global for output
    p95 = g['p95_global'].iloc[0] if 'p95_global' in g.columns else np.nan
    stats['p95_global'] = p95
    results.append(stats)

results_df = pd.DataFrame(results)

# Convert earliest epoch to Asia/Shanghai timestamp string
results_df['earliest_segment_start'] = results_df['earliest_segment_start_epoch'].apply(epoch_to_shanghai_str_safe)
results_df = results_df.drop(columns=['earliest_segment_start_epoch'])

# 3) Keep only groups where max_segment_length >= 2 (sustained fault segments)
sustained = results_df[results_df['max_segment_length'] >= 2].copy()

# Merge max_value_in_window and p95_global already present; ensure column order
sustained = sustained[[
    'cmdb_id','kpi_name','p95_global','total_points_in_window','total_anomaly_points',
    'num_anomaly_segments','max_segment_length','earliest_segment_start','max_value_in_window'
]]

# Sort by earliest_segment_start ascending (NA last) then max_segment_length descending
# For sorting, convert earliest_segment_start back to epoch where possible
def to_epoch_or_large_na(ts):
    if pd.isna(ts):
        return np.iinfo(np.int64).max
    try:
        return int(pd.to_datetime(ts).tz_localize(tz).tz_convert('UTC').timestamp())
    except Exception:
        return np.iinfo(np.int64).max

sustained['_sort_epoch'] = sustained['earliest_segment_start'].apply(lambda x: to_epoch_or_large_na(x))
sustained = sustained.sort_values(by=['_sort_epoch','max_segment_length'], ascending=[True, False]).reset_index(drop=True).drop(columns=['_sort_epoch'])

sustained_top20 = sustained.head(20).copy()

# 4) Summary counts
groups_scanned = int(group_stats.shape[0])
groups_with_any_anomaly = int((results_df['total_anomaly_points'] > 0).sum())
groups_with_sustained = int(sustained.shape[0])

# Store intermediates in kernel variables for later steps
group_stats_full_df = group_stats
metric_window_df = metric_window
results_full_df = results_df
sustained_full_df = sustained

# Final compact outputs
sustained_top20, groups_scanned, groups_with_any_anomaly, groups_with_sustained

```
Out[6]:
```
```python
# Build a concise plain-English summary using kernel results:
# Reuse existing variables: sustained_top20, groups_scanned, groups_with_any_anomaly, groups_with_sustained

def fmt_num(v):
    try:
        if pd.isna(v):
            return "<NA>"
        v = float(v)
        if abs(v - round(v)) < 1e-9:
            return str(int(round(v)))
        if abs(v) >= 1e6:
            return f"{v:,.0f}"
        return f"{v:.6g}"
    except Exception:
        return str(v)

lines = []
lines.append("Fault identification summary (incident window: 2021-03-09 15:30:00 → 2021-03-09 16:00:00 CST):")
lines.append("")
lines.append(f"- Groups scanned (IG01, IG02, Tomcat01-04): {groups_scanned}")
lines.append(f"- Groups with any anomaly points: {groups_with_any_anomaly}")
lines.append(f"- Groups with sustained fault segments (>=2 consecutive anomaly points): {groups_with_sustained}")
lines.append("")
lines.append("Sustained fault examples (groups with max_segment_length >= 2), showing earliest sustained start and severity:")
if 'sustained_top20' in globals() and not sustained_top20.empty:
    for _, r in sustained_top20.head(6).iterrows():
        cmdb = r['cmdb_id']
        kpi = (r['kpi_name'][:140] + '...') if len(str(r['kpi_name']))>140 else r['kpi_name']
        p95 = fmt_num(r['p95_global'])
        total_pts = fmt_num(r['total_points_in_window'])
        anom_pts = fmt_num(r['total_anomaly_points'])
        num_segs = fmt_num(r['num_anomaly_segments'])
        max_len = fmt_num(r['max_segment_length'])
        start = r['earliest_segment_start'] if pd.notna(r['earliest_segment_start']) else "<NA>"
        maxv = fmt_num(r['max_value_in_window'])
        lines.append(f" - {cmdb} | {kpi}")
        lines.append(f"     p95_global={p95}, pts_in_window={total_pts}, anomaly_pts={anom_pts}, segments={num_segs}, max_segment_length={max_len}")
        lines.append(f"     earliest_segment_start={start}, max_value_in_window={maxv}")
else:
    lines.append("  (no sustained faults found)")

lines.append("")
lines.append("Key observations:")
lines.append(" - Sustained faults are present on both IG and Tomcat components (16 groups total).")
lines.append(" - Many sustained groups are request-processing KPIs on Tomcat nodes and CPU/disk/network KPIs on IG/Tomcat.")
lines.append(" - Earliest sustained segments start at 2021-03-09 15:30:00 CST for several groups, indicating problems at window start.")
lines.append("")
lines.append("Recommended next steps:")
lines.append(" - Prioritize investigation of the listed sustained groups (start times and multi-point segments).")
lines.append(" - Correlate with trace-level latency spikes and application logs around the earliest_segment_start timestamps.")
lines.append(" - Check resource contention (CPU, disk I/O, network) on affected hosts and review recent deployments or batch jobs.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name    p95_global  total_points_in_window  total_anomaly_points  num_anomaly_segments  max_segment_length   earliest_segment_start  max_value_in_window
0       IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  2.500000e+00                      31                     2                     1                   2  2021-03-09 15:30:00 CST         4.200000e+01
1   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  3.065000e+02                      31                     2                     1                   2  2021-03-09 15:30:00 CST         3.070000e+02
2       IG02                             OSLinux-CPU_CPU_CPUWio  8.833920e-01                      31                     2                     1                   2  2021-03-09 15:32:00 CST         1.504700e+00
3       IG01                         OSLinux-CPU_CPU_CPUSysTime  5.344500e-01                      31                     2                     1                   2  2021-03-09 15:37:00 CST         5.533140e-01
4       IG01                        OSLinux-CPU_CPU_CPUUserTime  2.540035e+01                      31                     2                     1                   2  2021-03-09 15:37:00 CST         2.545659e+01
5       IG02                            OSLinux-CPU_CPU_CPULoad  1.315000e+00                      31                     2                     1                   2  2021-03-09 15:58:00 CST         1.700000e+00
6       IG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime  6.475742e+06                      31                     2                     1                   2  2021-03-09 15:59:00 CST         6.475833e+06
7       IG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  6.475789e+06                      31                     2                     1                   2  2021-03-09 15:59:00 CST         6.475878e+06
8   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  9.236032e+08                      31                     2                     1                   2  2021-03-09 15:59:00 CST         9.239711e+08
9   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  1.348760e+06                      31                     2                     1                   2  2021-03-09 15:59:00 CST         1.349462e+06
10  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.588547e+08                      31                     2                     1                   2  2021-03-09 15:59:00 CST         1.591993e+08
11  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  1.450845e+05                      31                     2                     1                   2  2021-03-09 15:59:00 CST         1.457570e+05
12  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.587001e+08                      31                     2                     1                   2  2021-03-09 15:59:00 CST         1.590840e+08
13  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  1.447185e+05                      31                     2                     1                   2  2021-03-09 15:59:00 CST         1.453680e+05
14  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.459153e+08                      31                     2                     1                   2  2021-03-09 15:59:00 CST         1.462192e+08
15  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  1.428630e+05                      31                     2                     1                   2  2021-03-09 15:59:00 CST         1.435000e+05, 617, 249, 16)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing 'trace' DataFrame from kernel
tz = pytz.timezone('Asia/Shanghai')

# Target components and time window
components = ["IG01","IG02","Tomcat01","Tomcat02","Tomcat03","Tomcat04"]
start_ts = int(pd.Timestamp("2021-03-09 15:30:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2021-03-09 16:00:00", tz=tz).tz_convert('UTC').timestamp())

# Ensure numeric types
trace['timestamp'] = pd.to_numeric(trace['timestamp'], errors='coerce').astype('Int64')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# 1) Select trace groups by name patterns for the target components (use full series for p95)
pattern_self = "trace.self.duration"
pattern_to_tomcat = ("trace.to_Tomcat01","trace.to_Tomcat02","trace.to_Tomcat03","trace.to_Tomcat04")

def matches_trace_name(n):
    if pd.isna(n):
        return False
    if pattern_self in n:
        return True
    for p in pattern_to_tomcat:
        if p in n:
            return True
    return False

mask_candidates_full = trace['cmdb_id'].isin(components) & trace['trace_name'].apply(matches_trace_name)
trace_selected_full = trace[mask_candidates_full].copy()

# Compute global p95 and total points per (cmdb_id, trace_name) using full series (rule: global before filtering)
selected_group_stats_full = trace_selected_full.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    total_points_full_series = ('value','count'),
    p95_global = ('value', lambda x: float(np.percentile(x.dropna(),95)) if x.dropna().size>0 else np.nan)
)

# 2) Filter trace rows to incident window and restrict to selected groups
trace_window = trace[(trace['timestamp'] >= start_ts) & (trace['timestamp'] <= end_ts)].copy()
trace_window_selected = trace_window[trace_window['cmdb_id'].isin(components) & trace_window['trace_name'].apply(matches_trace_name)].copy()

# Join p95_global onto window rows
trace_window_selected = trace_window_selected.merge(
    selected_group_stats_full[['cmdb_id','trace_name','p95_global']],
    on=['cmdb_id','trace_name'],
    how='left'
)

# Mark points above global p95 (handle NaN p95 as False)
trace_window_selected['above_p95'] = (~trace_window_selected['p95_global'].isna()) & (trace_window_selected['value'] > trace_window_selected['p95_global'])

# 2) Aggregations per group within window
agg_cols = trace_window_selected.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    count_points_in_window = ('value','count'),
    count_points_above_p95 = ('above_p95','sum'),
    max_value_in_window = ('value','max')
)

# earliest timestamp above p95 per group
mask_above = trace_window_selected[trace_window_selected['above_p95']].copy()
earliest_above = mask_above.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    earliest_epoch_above_p95 = ('timestamp','min')
)

# Merge aggregates with p95_global
agg_trace_selected = agg_cols.merge(selected_group_stats_full[['cmdb_id','trace_name','p95_global']], on=['cmdb_id','trace_name'], how='left')
agg_trace_selected = agg_trace_selected.merge(earliest_above, on=['cmdb_id','trace_name'], how='left')

# Convert earliest epoch to Asia/Shanghai string
def epoch_to_shanghai_str_safe(s):
    if pd.isna(s):
        return pd.NA
    return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

agg_trace_selected['earliest_timestamp_above_p95'] = agg_trace_selected['earliest_epoch_above_p95'].apply(epoch_to_shanghai_str_safe)
agg_trace_selected = agg_trace_selected.drop(columns=['earliest_epoch_above_p95'])

# 3) Keep only groups where count_points_above_p95 > 0
anomalies_trace_selected = agg_trace_selected[agg_trace_selected['count_points_above_p95'] > 0].copy()

# Sort by earliest_timestamp_above_p95 asc (NA last) then count_points_above_p95 desc
# For robust sort, create epoch sort key from the string where possible
def ts_to_epoch_or_large_na(ts):
    if pd.isna(ts):
        return np.iinfo(np.int64).max
    try:
        # parse string (already localized to Asia/Shanghai)
        return int(pd.to_datetime(ts).tz_convert('UTC').timestamp())
    except Exception:
        return np.iinfo(np.int64).max

anomalies_trace_selected['_sort_epoch'] = anomalies_trace_selected['earliest_timestamp_above_p95'].apply(lambda x: ts_to_epoch_or_large_na(x))
anomalies_trace_selected = anomalies_trace_selected.sort_values(by=['_sort_epoch','count_points_above_p95'], ascending=[True, False]).drop(columns=['_sort_epoch']).reset_index(drop=True)

# Prepare compact output table (limit to top 50)
anomalies_trace_selected_table = anomalies_trace_selected[
    ['cmdb_id','trace_name','p95_global','count_points_in_window','count_points_above_p95','earliest_timestamp_above_p95','max_value_in_window']
].head(50).copy()

# 4) Single-line summary identifying earliest anomaly among these groups
if not anomalies_trace_selected.empty:
    # find earliest by comparing earliest_timestamp_above_p95
    # convert to epoch to find min
    anomalies_trace_selected['earliest_epoch'] = anomalies_trace_selected['earliest_timestamp_above_p95'].apply(lambda x: ts_to_epoch_or_large_na(x))
    earliest_row = anomalies_trace_selected.loc[anomalies_trace_selected['earliest_epoch'].idxmin()]
    earliest_ts = earliest_row['earliest_timestamp_above_p95']
    earliest_comp = earliest_row['cmdb_id']
    earliest_trace = earliest_row['trace_name']
    # classify type
    trace_name_lower = str(earliest_trace)
    if "trace.self.duration" in trace_name_lower:
        if str(earliest_comp).startswith("IG"):
            earliest_type = "IG self-trace"
        elif str(earliest_comp).startswith("Tomcat"):
            earliest_type = "Tomcat self-trace"
        else:
            earliest_type = "self-trace"
    elif any(p in trace_name_lower for p in pattern_to_tomcat):
        earliest_type = "IG->Tomcat call"
    else:
        earliest_type = "other"
    earliest_summary = (
        f"Earliest anomaly at {earliest_ts} on {earliest_comp} | trace_name={earliest_trace} | type={earliest_type}"
    )
    # drop helper column
    anomalies_trace_selected = anomalies_trace_selected.drop(columns=['earliest_epoch'])
else:
    earliest_summary = "No anomalies detected in selected trace groups in the incident window."

# Store intermediate tables for later steps
selected_group_stats_full_df = selected_group_stats_full
trace_window_selected_full_df = trace_window_selected
agg_trace_selected_full_df = agg_trace_selected
anomalies_trace_selected_full_df = anomalies_trace_selected

# Final compact outputs
anomalies_trace_selected_table, earliest_summary

```
Out[7]:
```
```python
# Build a concise plain-English summary from previously computed trace anomaly results.
# Reuse kernel variables: selected_group_stats_full_df, anomalies_trace_selected_table, earliest_summary

import pandas as pd

# Safety checks and derive counts
total_selected_groups = int(selected_group_stats_full_df.shape[0]) if 'selected_group_stats_full_df' in globals() else 0
anomalous_groups = int(anomalies_trace_selected_table.shape[0]) if 'anomalies_trace_selected_table' in globals() else 0

# Aggregate anomalies by component for a compact highlight
if anomalous_groups > 0:
    comp_agg = (
        anomalies_trace_selected_table
        .groupby('cmdb_id', as_index=False)
        .agg(num_anomalous_trace_groups=('trace_name','count'),
             total_anomalous_points=('count_points_above_p95','sum'))
        .sort_values(['num_anomalous_trace_groups','total_anomalous_points'], ascending=[False, False])
    )
    # Create a short summary string for top components (limit 6)
    top_comp_lines = []
    for _, r in comp_agg.head(6).iterrows():
        top_comp_lines.append(f"{r['cmdb_id']}: {int(r['num_anomalous_trace_groups'])} trace groups, {int(r['total_anomalous_points'])} anomaly points")
    top_comp_summary = "; ".join(top_comp_lines)
else:
    comp_agg = pd.DataFrame()
    top_comp_summary = "None"

# Compose plain-English summary
lines = []
lines.append("Trace anomaly detection summary (selected trace types in incident window 2021-03-09 15:30→16:00 CST):")
lines.append(f"- Trace groups scanned (selected patterns across IG/Tomcat): {total_selected_groups}")
lines.append(f"- Trace groups with anomalies (value > group global P95): {anomalous_groups}")
lines.append("")
lines.append(f"- Earliest anomaly detected: {earliest_summary}")
lines.append("")
lines.append("- Top impacted components (by number of anomalous trace groups):")
lines.append(f"  {top_comp_summary}")
lines.append("")
lines.append("Key observations:")
lines.append(" - IG01 shows the very earliest anomalies (at 2021-03-09 15:30:00 CST) on self-duration and IG->Tomcat calls.")
lines.append(" - Multiple IG->Tomcat and trace.self.duration groups across IG01/IG02 and Tomcat nodes had points exceeding their global P95 (typically 1–2 points per group).")
lines.append(" - These anomalies align with trace-level latency spikes and should be investigated first on IG services and the Tomcat targets they call.")
lines.append("")
lines.append("Recommended next steps:")
lines.append(" - Prioritize IG01/IG02 and Tomcat nodes listed in the top components; correlate these trace anomalies with metric spikes and application logs at the earliest timestamps.")
lines.append(" - Inspect request paths from IG to Tomcat (latency and row_count increases) and look for upstream bottlenecks or transient resource contention.")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                       trace_name  p95_global  count_points_in_window  count_points_above_p95 earliest_timestamp_above_p95  max_value_in_window
0       IG01         trace.self.duration_mean    1.005696                      30                       2      2021-03-09 15:30:00 CST             1.579263
1       IG01          trace.self.duration_p95    4.176140                      30                       2      2021-03-09 15:30:00 CST             6.495000
2       IG01  trace.to_Tomcat01.duration_mean    1.006720                      30                       2      2021-03-09 15:30:00 CST             1.645598
3       IG01   trace.to_Tomcat01.duration_p95    4.190285                      30                       2      2021-03-09 15:30:00 CST             6.499750
4       IG01      trace.to_Tomcat01.row_count  294.650000                      30                       2      2021-03-09 15:31:00 CST           315.000000
..       ...                              ...         ...                     ...                     ...                          ...                  ...
31  Tomcat02          trace.self.duration_p95    1.035150                      30                       2      2021-03-09 15:30:00 CST             1.106000
32  Tomcat03         trace.self.duration_mean    0.153338                      30                       2      2021-03-09 15:30:00 CST             0.199442
33  Tomcat03          trace.self.duration_p95    1.036833                      30                       2      2021-03-09 15:30:00 CST             1.059650
34  Tomcat04         trace.self.duration_mean    0.146070                      30                       2      2021-03-09 15:30:00 CST             0.252233
35  Tomcat04          trace.self.duration_p95    1.036358                      30                       2      2021-03-09 15:30:00 CST             1.072000

[36 rows x 7 columns], 'Earliest anomaly at 2021-03-09 15:30:00 CST on IG01 | trace_name=trace.self.duration_mean | type=IG self-trace')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing DataFrame from kernel:
# anomalies_trace_selected_table (group-level selected trace anomalies table)
# anomalies_trace_selected_full_df (full anomalies table with helper fields, if present)

tz = pytz.timezone('Asia/Shanghai')

# Safety check
if 'anomalies_trace_selected_table' not in globals():
    raise RuntimeError("Expected 'anomalies_trace_selected_table' in kernel (from prior step).")

# Work on a copy to avoid mutating original unexpectedly
anomalies_trace_with_ratio = anomalies_trace_selected_table.copy()

# 1) Compute exceed_ratio = max_value_in_window / p95_global, handle p95_global == 0 as inf
def compute_exceed_ratio(row):
    p = row['p95_global']
    m = row['max_value_in_window']
    if pd.isna(p):
        return np.nan
    try:
        if p == 0:
            return np.inf
        return float(m) / float(p)
    except Exception:
        return np.nan

anomalies_trace_with_ratio['exceed_ratio'] = anomalies_trace_with_ratio.apply(compute_exceed_ratio, axis=1)

# Helper to convert earliest_timestamp_above_p95 string to epoch for sorting (NA -> large int)
def ts_to_epoch_or_max(ts_str):
    if pd.isna(ts_str):
        return np.iinfo(np.int64).max
    try:
        # Remove trailing timezone token if present (formatted as "... CST")
        # Then localize to Asia/Shanghai explicitly
        s = str(ts_str).replace(' CST','')
        dt = pd.to_datetime(s)
        if dt.tzinfo is None:
            dt = tz.localize(dt)
        else:
            # ensure in Asia/Shanghai
            dt = dt.tz_convert(tz)
        return int(dt.tz_convert('UTC').timestamp())
    except Exception:
        return np.iinfo(np.int64).max

# 2) Prepare compact group-level table sorted by exceed_ratio desc, then earliest_timestamp_above_p95 asc
anomalies_trace_with_ratio['_earliest_epoch_sort'] = anomalies_trace_with_ratio['earliest_timestamp_above_p95'].apply(ts_to_epoch_or_max)
anomalies_trace_with_ratio_sorted = anomalies_trace_with_ratio.sort_values(
    by=['exceed_ratio','_earliest_epoch_sort'],
    ascending=[False, True]
).drop(columns=['_earliest_epoch_sort']).reset_index(drop=True)

group_table_top50 = anomalies_trace_with_ratio_sorted[
    ['cmdb_id','trace_name','p95_global','count_points_in_window','count_points_above_p95',
     'earliest_timestamp_above_p95','max_value_in_window','exceed_ratio']
].head(50).copy()

# 3) Aggregate per component
# Use only rows in anomalies_trace_with_ratio (these already have count_points_above_p95>0)
comp_agg = anomalies_trace_with_ratio.groupby('cmdb_id', as_index=False).agg(
    num_anomalous_trace_groups = ('trace_name', 'nunique'),
    total_anomalous_trace_points = ('count_points_above_p95', 'sum'),
    earliest_epoch = ('earliest_timestamp_above_p95', lambda col: col.apply(ts_to_epoch_or_max).min()),
    component_max_exceed_ratio = ('exceed_ratio', 'max')
)

# Convert earliest_epoch back to Asia/Shanghai formatted string (handle max sentinel as NA)
def epoch_to_shanghai_str_safe(epoch):
    if epoch is None or (isinstance(epoch, (int, np.integer)) and epoch >= np.iinfo(np.int64).max):
        return pd.NA
    return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

comp_agg['earliest_anomaly_timestamp'] = comp_agg['earliest_epoch'].apply(epoch_to_shanghai_str_safe)
component_summary_sorted = comp_agg[[
    'cmdb_id','num_anomalous_trace_groups','total_anomalous_trace_points','earliest_anomaly_timestamp','component_max_exceed_ratio'
]].rename(columns={'cmdb_id':'component_name'}).sort_values(by='component_max_exceed_ratio', ascending=False).reset_index(drop=True)

# 4) Single-line recommendation: component with largest component_max_exceed_ratio
if not component_summary_sorted.empty:
    top_comp_row = component_summary_sorted.iloc[0]
    top_comp_name = top_comp_row['component_name']
    top_comp_ts = top_comp_row['earliest_anomaly_timestamp']
    recommendation = f"{top_comp_name} has the largest severity (component_max_exceed_ratio={top_comp_row['component_max_exceed_ratio']:.6g}); earliest anomaly at {top_comp_ts}"
else:
    recommendation = "No anomalous trace groups found; no recommendation."

# Store full intermediate tables for later steps
anomalies_trace_with_ratio_full_df = anomalies_trace_with_ratio
component_summary_full_df = comp_agg

# Final compact outputs (DataFrames + recommendation string)
group_table_top50, component_summary_sorted, recommendation

```
Out[8]:
```
```python
# Concise plain-English summary using previously computed tables:
# Reuse kernel variables: group_table_top50, component_summary_sorted, recommendation

# Prepare counts and top items from stored tables
n_trace_groups = int(group_table_top50.shape[0]) if 'group_table_top50' in globals() else 0
total_anom_points = int(group_table_top50['count_points_above_p95'].sum()) if 'group_table_top50' in globals() else 0

top_comp = component_summary_sorted.iloc[0] if ('component_summary_sorted' in globals() and not component_summary_sorted.empty) else None
top_comp_name = top_comp['component_name'] if top_comp is not None else "<none>"
top_comp_ratio = float(top_comp['component_max_exceed_ratio']) if top_comp is not None else None
top_comp_earliest = top_comp['earliest_anomaly_timestamp'] if top_comp is not None else "<NA>"

# Earliest anomaly row (from group_table_top50, which is sorted by exceed_ratio then time)
earliest_row = None
if 'group_table_top50' in globals() and not group_table_top50.empty:
    # find global earliest timestamp among these rows
    valid_times = group_table_top50[group_table_top50['earliest_timestamp_above_p95'].notna()].copy()
    if not valid_times.empty:
        earliest_idx = valid_times['earliest_timestamp_above_p95'].astype(str).sort_values().index[0]
        earliest_row = valid_times.loc[earliest_idx]

# Compose summary string
lines = []
lines.append(f"Trace anomaly severity summary (selected IG/Tomcat traces):")
lines.append(f"- Anomalous trace groups detected: {n_trace_groups}, total anomaly points in-window: {total_anom_points}.")
if earliest_row is not None:
    lines.append(f"- Earliest anomalous point: {earliest_row['earliest_timestamp_above_p95']} on {earliest_row['cmdb_id']} | {earliest_row['trace_name']}.")
else:
    lines.append("- No earliest anomalous point available.")
if top_comp is not None:
    lines.append(f"- Highest-severity component: {top_comp_name} (component_max_exceed_ratio ≈ {top_comp_ratio:.6g}), earliest anomaly at {top_comp_earliest}.")
else:
    lines.append("- No component severity data available.")
lines.append("- Key insight: IG01 and IG02 drive most and the largest exceedances (IG01 is the top priority). Many exceedances are on IG self-duration and IG→Tomcat duration/p95 traces.")
lines.append("- Recommendation: prioritize investigating IG services (start with IG01), focus on IG→Tomcat call latency and correlate with Tomcat-side traces/metrics around the earliest timestamps (2021-03-09 15:30:00 CST).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                       trace_name  p95_global  count_points_in_window  count_points_above_p95 earliest_timestamp_above_p95  max_value_in_window  exceed_ratio
0       IG01   trace.to_Tomcat04.duration_p95    4.161510                      30                       2      2021-03-09 15:30:00 CST             7.949500      1.910244
1       IG02   trace.to_Tomcat04.duration_p95    4.001552                      30                       2      2021-03-09 15:30:00 CST             7.254000      1.812796
2       IG02          trace.self.duration_p95    3.973942                      30                       2      2021-03-09 15:30:00 CST             6.888000      1.733291
3   Tomcat04         trace.self.duration_mean    0.146070                      30                       2      2021-03-09 15:30:00 CST             0.252233      1.726791
4       IG01  trace.to_Tomcat04.duration_mean    1.004549                      30                       2      2021-03-09 15:30:00 CST             1.712983      1.705226
..       ...                              ...         ...                     ...                     ...                          ...                  ...           ...
31  Tomcat02          trace.self.duration_p95    1.035150                      30                       2      2021-03-09 15:30:00 CST             1.106000      1.068444
32  Tomcat04          trace.self.duration_p95    1.036358                      30                       2      2021-03-09 15:30:00 CST             1.072000      1.034392
33      IG02      trace.to_Tomcat02.row_count  286.350000                      30                       2      2021-03-09 15:31:00 CST           296.000000      1.033700
34      IG02      trace.to_Tomcat04.row_count  304.050000                      30                       2      2021-03-09 15:31:00 CST           311.000000      1.022858
35  Tomcat03          trace.self.duration_p95    1.036833                      30                       2      2021-03-09 15:30:00 CST             1.059650      1.022007

[36 rows x 8 columns],   component_name  num_anomalous_trace_groups  total_anomalous_trace_points earliest_anomaly_timestamp  component_max_exceed_ratio
0           IG01                          14                            28    2021-03-09 15:30:00 CST                    1.910244
1           IG02                          14                            28    2021-03-09 15:30:00 CST                    1.812796
2       Tomcat04                           2                             4    2021-03-09 15:30:00 CST                    1.726791
3       Tomcat02                           2                             4    2021-03-09 15:30:00 CST                    1.684773
4       Tomcat01                           2                             4    2021-03-09 15:30:00 CST                    1.593002
5       Tomcat03                           2                             4    2021-03-09 15:30:00 CST                    1.300666, 'IG01 has the largest severity (component_max_exceed_ratio=1.91024); earliest anomaly at 2021-03-09 15:30:00 CST')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing DataFrames from kernel: log, error_logs
# Ensure required variables and types
components = ["IG01","IG02","Tomcat01","Tomcat02","Tomcat03","Tomcat04"]
tz = pytz.timezone('Asia/Shanghai')

# Normalize timestamp and value columns
if 'log' in globals():
    log['timestamp'] = pd.to_numeric(log['timestamp'], errors='coerce').astype('Int64')
    log['value'] = pd.to_numeric(log['value'], errors='coerce')
else:
    log = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])

if 'error_logs' in globals():
    error_logs['timestamp'] = pd.to_numeric(error_logs['timestamp'], errors='coerce').astype('Int64')
else:
    error_logs = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Incident window (Asia/Shanghai -> convert to epoch UTC)
start_ts = int(pd.Timestamp("2021-03-09 15:30:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2021-03-09 16:00:00", tz=tz).tz_convert('UTC').timestamp())

# Helper to format epoch -> Asia/Shanghai string
def epoch_to_shanghai_str_safe(s):
    if pd.isna(s):
        return pd.NA
    return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# 1) log.csv: compute global P95 and total points per (cmdb_id, log_name) for target components
log_target_full = log[log['cmdb_id'].isin(components)].copy()

if not log_target_full.empty:
    group_stats_log_full = log_target_full.groupby(['cmdb_id','log_name'], as_index=False).agg(
        total_points_full_series = ('value','count'),
        p95_global = ('value', lambda x: float(np.percentile(x.dropna(),95)) if x.dropna().size>0 else np.nan)
    )
else:
    group_stats_log_full = pd.DataFrame(columns=['cmdb_id','log_name','total_points_full_series','p95_global'])

# Filter log rows to incident window for target components
log_window = log_target_full[
    (log_target_full['timestamp'] >= start_ts) & (log_target_full['timestamp'] <= end_ts)
].copy()

# Merge p95_global onto window rows
if not log_window.empty and not group_stats_log_full.empty:
    log_window = log_window.merge(group_stats_log_full, on=['cmdb_id','log_name'], how='left')
    # Mark above p95 (handle NaN p95 as False)
    log_window['above_p95'] = (~log_window['p95_global'].isna()) & (log_window['value'] > log_window['p95_global'])
else:
    # Ensure columns exist
    log_window['total_points_full_series'] = pd.Series(dtype='Int64')
    log_window['p95_global'] = pd.Series(dtype='float')
    log_window['above_p95'] = pd.Series(dtype='bool')

# Aggregate per (cmdb_id, log_name) within window
if not log_window.empty:
    agg_log_window = log_window.groupby(['cmdb_id','log_name'], as_index=False).agg(
        count_points_in_window = ('value','count'),
        count_points_above_p95 = ('above_p95','sum'),
        max_value_in_window = ('value','max')
    )
    # earliest timestamp above p95
    mask_above_log = log_window[log_window['above_p95']].copy()
    if not mask_above_log.empty:
        earliest_above_log = mask_above_log.groupby(['cmdb_id','log_name'], as_index=False).agg(
            earliest_epoch_above_p95 = ('timestamp','min')
        )
        agg_log_window = agg_log_window.merge(earliest_above_log, on=['cmdb_id','log_name'], how='left')
    else:
        agg_log_window['earliest_epoch_above_p95'] = pd.NA
    # merge p95 and total_points_full_series for display
    agg_log_window = agg_log_window.merge(group_stats_log_full, on=['cmdb_id','log_name'], how='left')
    # convert earliest epoch to string
    if 'earliest_epoch_above_p95' in agg_log_window.columns:
        agg_log_window['earliest_timestamp_above_p95'] = agg_log_window['earliest_epoch_above_p95'].apply(epoch_to_shanghai_str_safe)
        agg_log_window = agg_log_window.drop(columns=['earliest_epoch_above_p95'])
else:
    agg_log_window = pd.DataFrame(columns=[
        'cmdb_id','log_name','count_points_in_window','count_points_above_p95','max_value_in_window',
        'total_points_full_series','p95_global','earliest_timestamp_above_p95'
    ])

# Keep only groups with anomalies (count_points_above_p95 > 0)
log_anomalies = agg_log_window[agg_log_window['count_points_above_p95'] > 0].copy()

# Sort by earliest_timestamp_above_p95 asc then count_points_above_p95 desc
def ts_to_epoch_or_max(ts):
    if pd.isna(ts):
        return np.iinfo(np.int64).max
    try:
        s = str(ts).replace(' CST','')
        dt = pd.to_datetime(s)
        if dt.tzinfo is None:
            dt = tz.localize(dt)
        else:
            dt = dt.tz_convert(tz)
        return int(dt.tz_convert('UTC').timestamp())
    except Exception:
        return np.iinfo(np.int64).max

if not log_anomalies.empty:
    log_anomalies['_sort_epoch'] = log_anomalies['earliest_timestamp_above_p95'].apply(ts_to_epoch_or_max)
    log_anomalies = log_anomalies.sort_values(by=['_sort_epoch','count_points_above_p95'], ascending=[True, False]).drop(columns=['_sort_epoch']).reset_index(drop=True)

# Prepare compact output columns and limit to top 20
log_anomalies_top20 = log_anomalies[
    ['cmdb_id','log_name','p95_global','total_points_full_series','count_points_in_window',
     'count_points_above_p95','earliest_timestamp_above_p95','max_value_in_window']
].head(20).copy()

# Counts
log_groups_scanned = int(group_stats_log_full.shape[0])
log_groups_with_anomalies = int(log_anomalies.shape[0])

# Store intermediates for later steps
group_stats_log_full_df = group_stats_log_full
log_window_full_df = log_window
agg_log_window_full_df = agg_log_window
log_anomalies_full_df = log_anomalies

# 2) error_logs.csv: total rows for these components and up to top 20 rows inside the incident window
error_target = error_logs[error_logs['cmdb_id'].isin(components)].copy()

if error_target.empty:
    error_info = pd.DataFrame([{
        'total_error_rows_for_components': 0,
        'note': 'error_logs.csv has zero rows for the target components'
    }])
    error_top20 = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
else:
    total_error_rows_for_components = int(error_target.shape[0])
    # rows inside incident window
    error_window = error_target[(error_target['timestamp'] >= start_ts) & (error_target['timestamp'] <= end_ts)].copy()
    # convert timestamp to Asia/Shanghai string for display
    if not error_window.empty:
        error_window['timestamp_sh'] = error_window['timestamp'].apply(epoch_to_shanghai_str_safe)
        error_top20 = error_window.sort_values('timestamp').loc[:, ['timestamp_sh','cmdb_id','message']].rename(columns={'timestamp_sh':'timestamp'}).head(20).copy()
    else:
        error_top20 = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
    error_info = pd.DataFrame([{
        'total_error_rows_for_components': total_error_rows_for_components,
        'rows_in_incident_window': int(error_window.shape[0])
    }])

# Store error intermediates
error_target_full_df = error_target
error_window_full_df = error_window if 'error_window' in locals() else pd.DataFrame()

# Final compact outputs (DataFrames and counts)
log_anomalies_top20, log_groups_scanned, log_groups_with_anomalies, error_info, error_top20

```
Out[9]:
```
```python
# Plain-English summary of log and error_log inspection for IG01, IG02, Tomcat01-04
summary = (
    "Log & error-log inspection (incident window 2021-03-09 15:30→16:00 CST):\n\n"
    "1) Log anomalies (computed vs global P95):\n"
    "   - Tomcat row counts: Tomcat01, Tomcat03, Tomcat04 each had 2 points above their p95 in-window\n"
    "     (Tomcat01 earliest 15:30; Tomcat03 and Tomcat04 earliest 15:31). This indicates elevated request volume.\n"
    "   - GC pause spikes: Tomcat04 had 2 above-p95 GC pauses (earliest 15:35); Tomcat01 and Tomcat03 had single above-p95 GC pauses\n"
    "     later in the window (Tomcat01 at ~15:47, Tomcat03 at ~15:53). This suggests transient GC activity on Tomcat nodes.\n\n"
    "   - Summary counts: scanned 12 log groups for these components; 6 groups had anomalies.\n\n"
    "2) Error logs: no error_log rows for these components inside the incident window (total rows for components = 0).\n\n"
    "Interpretation & recommended next steps:\n"
    " - Tomcat nodes show both increased request counts and GC pause spikes during the incident window. Prioritize Tomcat01/03/04 for investigation.\n"
    " - Correlate the row_count increases with trace latency spikes and the metric anomalies (CPU/disk/threads) around 15:30–15:40.\n"
    " - Investigate GC tuning and heap activity on Tomcat04 (earliest GC spike at ~15:35) and Tomcat01/03 later in the window.\n"
    " - Since error_logs contain no entries, focus on resource/latency diagnostics (metrics + traces) rather than application exceptions.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id          log_name  p95_global  total_points_full_series  count_points_in_window  count_points_above_p95 earliest_timestamp_above_p95  max_value_in_window
0  Tomcat01     log.row_count    581.0000                        31                      31                       2      2021-03-09 15:30:00 CST               634.00
1  Tomcat03     log.row_count    581.0000                        31                      31                       2      2021-03-09 15:31:00 CST               657.00
2  Tomcat04     log.row_count    594.5000                        31                      31                       2      2021-03-09 15:31:00 CST               635.00
3  Tomcat04  log.gc_pause_max      0.0985                        24                      24                       2      2021-03-09 15:35:00 CST                 0.12
4  Tomcat01  log.gc_pause_max      0.0900                        24                      24                       1      2021-03-09 15:47:00 CST                 0.10
5  Tomcat03  log.gc_pause_max      0.0900                        24                      24                       1      2021-03-09 15:53:00 CST                 0.12, 12, 6,    total_error_rows_for_components                                               note
0                                0  error_logs.csv has zero rows for the target co..., Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```